### Import Data

In [30]:
%run  Data_preparation_TCGA.ipynb
## More information about the dataset can be found in Data_preparation_TCGA.ipynb

### Import Graphs

In [31]:
import pickle

with open("random_graphs/top10_graphs.pkl", "rb") as f:
    top10_graphs = pickle.load(f)


with open("random_graphs/random_graphs.pkl", "rb") as f:
    random_graphs = pickle.load(f)

### Import the Model

In [ ]:
from Classification_model import *

### Training Process

In [70]:
train_loader = DataLoader(training_set, batch_size=1024, shuffle=True)
test_loader = DataLoader(testing_set, batch_size=5096, shuffle=False)

In [71]:
torch.manual_seed(0)

num_hiddens_genotype = 16
num_hiddens_final = 16

model = Drugcell_Vae(term_size_map, term_direct_gene_map, dG, num_genes, 
                 root, num_hiddens_genotype, num_hiddens_final, n_class = len(cancer_2_idx))

In [72]:
def create_term_mask(term_direct_gene_map, gene_dim, device):
    
    term_mask_map = {}

    for term, gene_set in term_direct_gene_map.items():

        mask = torch.zeros(len(gene_set), gene_dim)

        for i, gene_id in enumerate(gene_set):
            mask[i, gene_id] = 1

        mask_gpu = torch.autograd.Variable(mask)

        term_mask_map[term] = mask_gpu.to(device)

    return term_mask_map

term_mask_map = create_term_mask(model.term_direct_gene_map, num_genes, device = DEVICE)


In [73]:
def freeze_term_modules(model):
    for name, param in model.named_parameters():
        if (
            any(x in name for x in [
                '_linear_layer', 
                '_batchnorm_layer', 
            ])
            and 'final' not in name
        ):
            param.requires_grad = False


def unfreeze_term_modules(model):
    for name, param in model.named_parameters():
        if (
            any(x in name for x in [
                '_linear_layer', 
                '_batchnorm_layer', 
                '_aux_linear_layer1', 
                '_aux_linear_layer2',
                '_aux_linear_layer3',
                '_aux_linear_layer4'
            ])
            and 'final' not in name
        ):
            param.requires_grad = True

def get_trainable_params(model):
    return [p for p in model.parameters() if p.requires_grad]

In [74]:
learning_rate = 0.003
torch.manual_seed(0)
loss_list = []
accu_list = []
train_epochs = 500


for i, (score, dG) in enumerate(top10_graphs, 1):

    model = Drugcell_Vae(term_size_map, term_direct_gene_map, dG, num_genes, 
                 root, num_hiddens_genotype, num_hiddens_final, n_class = len(cancer_2_idx))
    model.to(DEVICE)

    model_loaded = torch.load('model_032_updated.pt', map_location='cpu')
    
    state_dict = model_loaded.state_dict()

    current_state_dict = model.state_dict()

    filtered_state_dict = {
        k: v for k, v in state_dict.items()
        if k in current_state_dict and v.size() == current_state_dict[k].size()
    }

    model.load_state_dict(filtered_state_dict, strict=False)

    freeze_term_modules(model)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, betas=(0.9, 0.99), eps=1e-05, weight_decay= 1e-4)


    optimizer.zero_grad()

    best_epoch = 0
    best_accu = 0
    count = 0
    best_model_path = "model_classification_freeze_encoder_random.pt"

    for name, param in model.named_parameters():
        term_name = name.split('_')[0]

        if '_direct_gene_layer.weight' in name:
            param.data = torch.mul(param.data, term_mask_map[term_name].to(DEVICE)) * 1
        else:
            param.data = param.data * 1

    tepoch = tqdm.tqdm(range(train_epochs))
    for epoch in tepoch:

        # Train
        model.train()
        train_predict = torch.zeros(0, 0).to(DEVICE)

        for i, (data, labels) in enumerate(train_loader):
            # Convert torch tensor to Variable

            # Forward + Backward + Optimize
            optimizer.zero_grad()  # zero the gradient buffer

            # Here term_NN_out_map is a dictionary
            logits, mu, log_var, aux_out_map, aux_cancer_map, term_NN_out_map, term_original_children = model(data.to(DEVICE))
            
            student_feats = term_NN_out_map

            if train_predict.size()[0] == 0:
                train_predict = aux_out_map["final"].data
            else:
                train_predict = torch.cat([train_predict, aux_out_map["final"].data], dim=0)

            total_loss = 0

            loss_vae, class_loss, KLD = model.loss_log_vae(
                logits=logits, y=labels.to(DEVICE), mu=mu, log_var=log_var, beta=0.001
            )

            loss_intermidiate = model.intermediate_loss_cancer(aux_cancer_map, labels.to(DEVICE))

            total_loss = torch.mean(loss_vae)

            tmp_loss = total_loss.item()
            
            total_loss.backward()

            for name, param in model.named_parameters():
                if "_direct_gene_layer.weight" not in name:
                    continue
                term_name = name.split("_")[0]
                # print name, param.grad.data.size(), term_mask_map[term_name].size()
                if param.requires_grad and param.grad is not None:
                    term_name = name.split("_")[0]
                    param.grad.data = torch.mul(param.grad.data, term_mask_map[term_name])

            optimizer.step()
        
        model.eval()
        with torch.no_grad():
            (inputdata, labels) = next(iter(test_loader))
            inputdata = inputdata.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            logits, mu, log_var, aux_out_map, aux_cancer_map, term_NN_out_map, term_original_children = model(inputdata)
            preds = torch.argmax(logits, dim=1)
            accu = (preds == labels).float().mean().item()
        
            loss_list.append(tmp_loss)
            accu_list.append(accu)

        # if epoch % 10 == 0:
        if accu > best_accu:
            count += 1
        else:
            count = 0
        
        if count == 5:
            count = 0
            best_epoch = epoch
            best_accu = accu
        tepoch.set_postfix({"Epoch": epoch, "Loss": tmp_loss, "Accuracy": accu})
            
    print(f"Epoch {best_epoch}: New best model saved with accuracy {best_accu:.4f}")


/tmp/ipykernel_2675035/3266380450.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_loaded = torch.load('model_032_updated.pt', map_location='cpu')
100%|██████████|

Epoch 384: New best model saved with accuracy 0.8338
Training complete. Best model saved at: model_classification_freeze_encoder_random.pt


/tmp/ipykernel_2675035/3266380450.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_loaded = torch.load('model_032_updated.pt', map_location='cpu')
100%|██████████|

Epoch 317: New best model saved with accuracy 0.8399
Training complete. Best model saved at: model_classification_freeze_encoder_random.pt


100%|██████████| 500/500 [15:44<00:00,  1.89s/it, Epoch=499, Loss=10.4, Accuracy=0.801]


...

Epoch 306: New best model saved with accuracy 0.8097
Training complete. Best model saved at: model_classification_freeze_encoder_random.pt


100%|██████████| 500/500 [15:14<00:00,  1.83s/it, Epoch=499, Loss=10.4, Accuracy=0.804]


Epoch 386: New best model saved with accuracy 0.8157
Training complete. Best model saved at: model_classification_freeze_encoder_random.pt


100%|██████████| 500/500 [15:14<00:00,  1.83s/it, Epoch=499, Loss=10.4, Accuracy=0.807]

Epoch 265: New best model saved with accuracy 0.8066
Training complete. Best model saved at: model_classification_freeze_encoder_random.pt


In [75]:
learning_rate = 0.003
torch.manual_seed(0)
loss_list = []
accu_list = []
train_epochs = 500


for i, (score, dG) in enumerate(random_graphs, 1):

    model = Drugcell_Vae(term_size_map, term_direct_gene_map, dG, num_genes, 
                 root, num_hiddens_genotype, num_hiddens_final, n_class = len(cancer_2_idx))
    model.to(DEVICE)

    model_loaded = torch.load('model_032_updated.pt', map_location='cpu')

    freeze_term_modules(model)
    
    state_dict = model_loaded.state_dict()

    current_state_dict = model.state_dict()

    filtered_state_dict = {
        k: v for k, v in state_dict.items()
        if k in current_state_dict and v.size() == current_state_dict[k].size()
    }

    model.load_state_dict(filtered_state_dict, strict=False)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, betas=(0.9, 0.99), eps=1e-05, weight_decay= 1e-4)


    optimizer.zero_grad()

    best_epoch = 0
    best_accu = 0
    count = 0
    best_model_path = "model_classification_freeze_encoder_random.pt"

    for name, param in model.named_parameters():
        term_name = name.split('_')[0]

        if '_direct_gene_layer.weight' in name:
            param.data = torch.mul(param.data, term_mask_map[term_name].to(DEVICE)) * 1
        else:
            param.data = param.data * 1

    tepoch = tqdm.tqdm(range(train_epochs))
    for epoch in tepoch:

        # Train
        model.train()
        train_predict = torch.zeros(0, 0).to(DEVICE)

        for i, (data, labels) in enumerate(train_loader):
            # Convert torch tensor to Variable

            # Forward + Backward + Optimize
            optimizer.zero_grad()  # zero the gradient buffer

            # Here term_NN_out_map is a dictionary
            logits, mu, log_var, aux_out_map, aux_cancer_map, term_NN_out_map, term_original_children = model(data.to(DEVICE))
            
            student_feats = term_NN_out_map

            if train_predict.size()[0] == 0:
                train_predict = aux_out_map["final"].data
            else:
                train_predict = torch.cat([train_predict, aux_out_map["final"].data], dim=0)

            total_loss = 0

            loss_vae, class_loss, KLD = model.loss_log_vae(
                logits=logits, y=labels.to(DEVICE), mu=mu, log_var=log_var, beta=0.001
            )

            loss_intermidiate = model.intermediate_loss_cancer(aux_cancer_map, labels.to(DEVICE))

            total_loss = torch.mean(loss_vae)

            tmp_loss = total_loss.item()
            
            total_loss.backward()

            for name, param in model.named_parameters():
                if "_direct_gene_layer.weight" not in name:
                    continue
                term_name = name.split("_")[0]
                # print name, param.grad.data.size(), term_mask_map[term_name].size()
                if param.requires_grad and param.grad is not None:
                    term_name = name.split("_")[0]
                    param.grad.data = torch.mul(param.grad.data, term_mask_map[term_name])

            optimizer.step()
        
        model.eval()
        with torch.no_grad():
            (inputdata, labels) = next(iter(test_loader))
            inputdata = inputdata.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            logits, mu, log_var, aux_out_map, aux_cancer_map, term_NN_out_map, term_original_children = model(inputdata)
            preds = torch.argmax(logits, dim=1)
            accu = (preds == labels).float().mean().item()
        
            loss_list.append(tmp_loss)
            accu_list.append(accu)

        # if epoch % 10 == 0:
        if accu > best_accu:
            count += 1
        else:
            count = 0
        
        if count == 5:
            count = 0
            best_epoch = epoch
            best_accu = accu
        tepoch.set_postfix({"Epoch": epoch, "Loss": tmp_loss, "Accuracy": accu})
            
    print(f"Epoch {best_epoch}: New best model saved with accuracy {best_accu:.4f}")


/tmp/ipykernel_2675035/474766090.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_loaded = torch.load('model_032_updated.pt', map_location='cpu')
100%|██████████| 

Epoch 462: New best model saved with accuracy 0.8429
Training complete. Best model saved at: model_classification_freeze_encoder_random.pt


100%|██████████| 500/500 [15:54<00:00,  1.91s/it, Epoch=499, Loss=10.4, Accuracy=0.825]


Epoch 480: New best model saved with accuracy 0.8429
Training complete. Best model saved at: model_classification_freeze_encoder_random.pt


100%|██████████| 500/500 [15:59<00:00,  1.92s/it, Epoch=499, Loss=10.4, Accuracy=0.834]


...

Epoch 407: New best model saved with accuracy 0.8671
Training complete. Best model saved at: model_classification_freeze_encoder_random.pt


100%|██████████| 500/500 [16:05<00:00,  1.93s/it, Epoch=499, Loss=10.4, Accuracy=0.822]


Epoch 329: New best model saved with accuracy 0.8308
Training complete. Best model saved at: model_classification_freeze_encoder_random.pt


100%|██████████| 500/500 [15:53<00:00,  1.91s/it, Epoch=499, Loss=10.4, Accuracy=0.864]

Epoch 275: New best model saved with accuracy 0.8489
Training complete. Best model saved at: model_classification_freeze_encoder_random.pt
